# Underwater vessel noise — reproducibility notebook

**Zindi research write-up, reproducibility appendix.**

This notebook re-derives, from the raw acoustic exports in `dataset/sa_full/acoustics/`
and nothing else, every quantitative claim in the write-up that is *not* a learned
parameter: the per-site low-frequency exceedance rate at 6 dB, the threshold
sensitivity sweep, the site x month pressure map (including its bootstrap interval),
the two figures that carry the argument, and the test-set moment solution behind the
optimal-constant submission. It then re-reads one already-submitted prediction vector
and computes a counterfactual that is explicitly **not** submitted.

### Ground rules of this notebook

* **Offline.** No network access, no downloads, no API calls. The only inputs are
  `dataset/` (raw data + competition CSVs) and a handful of small committed artifacts.
* **Committed artifacts are read-only.** `out/*.csv` and every `submission_*.csv`
  are opened for reading only, and are used as *reference targets* for assertions.
  The provenance guard in the final cell re-hashes all of them and would abort if a
  single byte had changed.
* **Every number that matters is asserted**, not eyeballed. The assertions are
  one-sided in the sense that they fail loudly on any drift larger than `1e-9`.
* **Outputs are written only into `PACKAGE/`**: the regenerated figures and the
  optimal-constant submission go to `PACKAGE/figs_regen/` and `PACKAGE/`, never back
  into `out/`, `figs/`, or an existing `submission_*.csv`.

The plain-python twin of this notebook is `PACKAGE/uw_reproduce.py`; it contains the
same cells in the same order (jupytext "light" format), so a reader without Jupyter
can run `python3 PACKAGE/uw_reproduce.py` and get the same numbers.

## 1. Setup, dependency versions, and the band definitions

The scientific content of the study rests on a specific set of 25 third-octave bands
that are common to all ten sites (31.6 Hz to 7943.3 Hz). Two disjoint groups are used
throughout:

* **low group** — the 15 bands at or below 794.3 Hz, where vessel-radiated noise
  dominates;
* **high group** — the 3 bands at or above 5011.9 Hz (5011.9, 6309.6, 7943.3 Hz — the
  only members of the 25-band set above 5 kHz), used as a *negative control*: if the
  low band were merely picking up a generic recording artefact, the control band
  should behave the same way.

(The control group is three bands, not five: the common-band grid is coarse at the top
end, and 5011.9 Hz is the third-from-last band. The group definition below is asserted,
so this is checked rather than assumed.)

The strings below are asserted against the canonical tuple, so a typo in a band name
(which would silently select the wrong column) aborts the notebook.

In [ ]:
import hashlib
import json
import math
import os
import sys
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import matplotlib
matplotlib.use("Agg")          # headless: this notebook only writes PNGs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

T_START = time.perf_counter()


def find_root():
    """Locate the project checkout: the directory that holds dataset/ and out/."""
    candidates = []
    env = os.environ.get("UW_ROOT")
    if env:
        candidates.append(Path(env))
    here = Path.cwd().resolve()
    candidates += [here, *here.parents]
    try:                                    # defined when run as a plain script
        candidates.append(Path(__file__).resolve().parent.parent)
    except NameError:                       # not defined inside a notebook kernel
        pass
    for cand in candidates:
        if (cand / "dataset" / "sa_full" / "acoustics").is_dir() and (cand / "out" / "events.csv").is_file():
            return cand.resolve()
    raise SystemExit(
        "Could not locate the project root. Run this from inside the checkout "
        "(the directory containing dataset/sa_full/acoustics and out/events.csv), "
        "or set UW_ROOT to it."
    )


ROOT = find_root()
DATA = ROOT / "dataset" / "sa_full" / "acoustics"
OUT = ROOT / "out"                     # committed artifacts: READ-ONLY
PKG = ROOT / "PACKAGE"
FIGDIR = PKG / "figs_regen"
FIGDIR.mkdir(parents=True, exist_ok=True)

print("python      ", sys.version.split()[0], "|", sys.platform)
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
try:
    import pyarrow
    print("pyarrow     ", pyarrow.__version__)
except PackageNotFoundError:            # pragma: no cover - pyarrow is required by read_parquet
    print("pyarrow      missing (parquet reading will fail)")
print("matplotlib  ", matplotlib.__version__)
for _pkg in ("nbformat", "nbclient"):
    try:
        print(f"{_pkg:<12}", version(_pkg))
    except PackageNotFoundError:
        print(f"{_pkg:<12} not installed")
print()
print("project root", ROOT)
print("raw data    ", DATA)
print("reference   ", OUT, "(read-only)")
print("outputs     ", FIGDIR)

### Band definitions and pipeline constants

All thresholds and windows are declared once, here, and then used everywhere. They are
the values the write-up reports:

| constant | value | meaning |
|---|---|---|
| `WIN` | 181 min | centred rolling-median window (~3 h) forming the local baseline |
| `MIN_PERIODS` | 60 | a baseline is defined only where at least 60 minutes of the window exist |
| `THRESH` | 6.0 dB | an "event minute" is a minute at least 6 dB above its own baseline |
| `DUR_MIN` | 55 s | keep only near-complete minutes (the dictionary reports 16,896 rows below 55 s) |
| `MIN_ROWS` | 500 | a site with fewer usable minutes is not reported at all |
| `SWEEP` | 4, 5, 6, 8, 10 dB | thresholds used for the sensitivity analysis |
| `BOOT_B` / `BOOT_SEED` | 400 / 0 | bootstrap replicates and seed for the monthly interval |

In [ ]:
BAND_STR = (
    "31.6", "39.8", "50.1", "63.1", "79.4", "100", "125.9", "158.5", "199.5",
    "251.2", "316.2", "398.1", "501.2", "631", "794.3", "1000", "1258.9",
    "1584.9", "1995.3", "2511.9", "3162.3", "3981.1", "5011.9", "6309.6", "7943.3",
)
COMMON = list(BAND_STR)
LOW = [b for b in COMMON if float(b) <= 794.3]
HIGH = [b for b in COMMON if float(b) >= 5011.9]

WIN, MIN_PERIODS = 181, 60
THRESH, DUR_MIN, MIN_ROWS = 6.0, 55.0, 500
SWEEP = (4.0, 5.0, 6.0, 8.0, 10.0)
BOOT_B, BOOT_SEED = 400, 0
MONTH_MIN_ROWS = 50                    # a month needs this many valid minutes for a rate

# The band definitions are load-bearing: assert them rather than trust them.
assert len(COMMON) == 25 and len(set(COMMON)) == 25, "expected 25 distinct bands"
assert [float(b) for b in COMMON] == sorted(float(b) for b in COMMON), "bands must be ascending"
assert COMMON[0] == "31.6" and COMMON[-1] == "7943.3", "band endpoints must be 31.6 and 7943.3 Hz"
assert LOW == COMMON[:15] and len(LOW) == 15, "low group must be the 15 bands <= 794.3 Hz"
assert LOW[-1] == "794.3" and LOW[0] == "31.6"
assert HIGH == COMMON[-3:] and len(HIGH) == 3, "high (control) group must be the 3 bands >= 5011.9 Hz"
assert HIGH[0] == "5011.9" and HIGH == ["5011.9", "6309.6", "7943.3"]
assert set(LOW).isdisjoint(HIGH) and set(LOW) | set(HIGH) <= set(COMMON)
assert [f"{b:g}" for b in (WIN, MIN_PERIODS, THRESH)] == ["181", "60", "6"], "constants drifted"
print("bands      :", len(COMMON), "common |", len(LOW), "low (<=794.3 Hz) |", len(HIGH), "control (>=5011.9 Hz)")
print("low        :", ", ".join(LOW))
print("control    :", ", ".join(HIGH))
print("constants  : WIN=%d  MIN_PERIODS=%d  THRESH=%.1f dB  DUR_MIN=%.0f s  MIN_ROWS=%d  SWEEP=%s"
      % (WIN, MIN_PERIODS, THRESH, DUR_MIN, MIN_ROWS, list(SWEEP)))

### Provenance guard

Before touching anything, hash the artifacts that must not change: the committed
result tables in `out/`, the committed figures in `figs/`, and every
`submission_*.csv` at the top level. The final cell re-hashes them and asserts the
digests are identical, which is the machine-checkable form of "read-only".

In [ ]:
def fingerprint(paths):
    return {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}


READONLY = (
    sorted(OUT.glob("*.csv"))
    + sorted((ROOT / "figs").glob("*"))
    + sorted(ROOT.glob("submission_*.csv"))
)
FP_BEFORE = fingerprint(READONLY)
print(f"guarded {len(FP_BEFORE)} read-only files "
      f"({sum(p.stat().st_size for p in READONLY) / 1e6:.1f} MB):")
for _grp, _label in ((sorted(OUT.glob("*.csv")), "out/"),
                     (sorted((ROOT / "figs").glob("*")), "figs/"),
                     (sorted(ROOT.glob("submission_*.csv")), "submission_*")):
    print(f"  {_label:<14} {len(_grp):>3} files")

## 2. Loading the ten site files and the 6 dB exceedance rate

### The pipeline

For each site parquet, in this order (the order matters: the rolling baseline is
computed on the *time-sorted* series, so sorting has to happen before anything else):

1. parse `t` with `errors='coerce'` and drop unparseable timestamps;
2. sort by `t`;
3. keep only rows with `dur >= 55` s, so every row is a near-complete minute and the
   "share of minutes" denominator is meaningful;
4. drop rows with a missing value in *any* of the 25 common bands, so the low and the
   control series are computed over exactly the same minutes;
5. skip the site entirely if fewer than 500 rows survive.

A **band group level** is the row-wise mean of its member `TOL_<band>` columns. The
**baseline** is a centred 181-minute rolling median of that level (at least 60 minutes
of window required), and the **excess** is `level - baseline`. A minute is an
**event minute** when the excess is at least 6 dB. The site's **rate** is the share of
valid minutes (those with a defined baseline) that are event minutes.

Everything downstream — the sweep, the monthly map, both figures — is a re-slicing of
the two excess series computed here, so the ten files are read exactly once.

In [ ]:
def load_site(path):
    """Read one site parquet and apply the cleaning chain. Returns (frame, meta)."""
    df = pd.read_parquet(path)
    meta = {"rows_raw": len(df)}
    df["t"] = pd.to_datetime(df["t"], errors="coerce")
    df = df.dropna(subset=["t"])
    meta["rows_ts"] = len(df)
    df = df.sort_values("t")
    df = df[df["dur"] >= DUR_MIN]
    meta["rows_dur"] = len(df)
    df = df.dropna(subset=[f"TOL_{b}" for b in COMMON])
    meta["rows_used"] = len(df)
    return df, meta


def band_excess(df, bands):
    """Row-wise mean level of a band group, its rolling-median baseline, and the excess."""
    level = df[[f"TOL_{b}" for b in bands]].mean(axis=1).to_numpy(dtype="float64")
    baseline = (
        pd.Series(level)
        .rolling(WIN, center=True, min_periods=MIN_PERIODS)
        .median()
        .to_numpy(dtype="float64")
    )
    excess = level - baseline
    valid = ~np.isnan(excess)
    return excess, valid


def rate_at(excess, valid, threshold):
    """Share of valid minutes whose excess reaches `threshold`."""
    event = (excess >= threshold) & valid
    return float(event[valid].mean())


SITES = []
print(f"{'site':<20}{'raw':>9}{'ts ok':>9}{'dur>=55':>9}{'used':>9}{'low 6dB':>9}{'ctl 6dB':>9}")
for path in sorted(DATA.glob("*.parquet")):
    site = path.stem
    df, meta = load_site(path)
    if len(df) < MIN_ROWS:
        print(f"{site:<20}{meta['rows_raw']:>9}{meta['rows_ts']:>9}{meta['rows_dur']:>9}"
              f"{len(df):>9}  -- skipped: fewer than {MIN_ROWS} usable rows")
        continue
    lo_excess, lo_valid = band_excess(df, LOW)
    hi_excess, hi_valid = band_excess(df, HIGH)
    rec = dict(
        site=site,
        n=int(lo_valid.sum()),
        t=df["t"].to_numpy(),
        hours=df["t"].dt.hour.to_numpy(),
        lo_excess=lo_excess, lo_valid=lo_valid,
        hi_excess=hi_excess, hi_valid=hi_valid,
        meta=meta,
    )
    SITES.append(rec)
    print(f"{site:<20}{meta['rows_raw']:>9}{meta['rows_ts']:>9}{meta['rows_dur']:>9}"
          f"{rec['n']:>9}{rate_at(lo_excess, lo_valid, THRESH):>8.2%}"
          f"{rate_at(hi_excess, hi_valid, THRESH):>9.2%}")
    del df

assert SITES, "no site survived the cleaning chain"
assert [s["site"] for s in SITES] == sorted(s["site"] for s in SITES), "site order must be filename order"
print()
print(f"{len(SITES)} sites kept | {sum(s['n'] for s in SITES):,} valid minutes in total")

### Assertion against the committed `out/events.csv`

The required columns are `site`, `n`, `lo_rate`, `hi_rate`; the committed file also
carries the modal event hour for each group (`lo_hour_peak`, `hi_hour_peak`), which is
cheap to reproduce and is asserted here too. Rates are compared with an absolute
tolerance of `1e-9` — roughly seven orders of magnitude tighter than the differences
that would matter scientifically, and far below the printed precision of the paper.

In [ ]:
events = pd.DataFrame([
    dict(
        site=s["site"],
        n=s["n"],
        lo_rate=rate_at(s["lo_excess"], s["lo_valid"], THRESH),
        hi_rate=rate_at(s["hi_excess"], s["hi_valid"], THRESH),
        lo_hour_peak=(
            int(pd.Series(s["hours"][(s["lo_excess"] >= THRESH) & s["lo_valid"]]).mode().iloc[0])
            if int(((s["lo_excess"] >= THRESH) & s["lo_valid"]).sum()) > 5 else -1
        ),
        hi_hour_peak=(
            int(pd.Series(s["hours"][(s["hi_excess"] >= THRESH) & s["hi_valid"]]).mode().iloc[0])
            if int(((s["hi_excess"] >= THRESH) & s["hi_valid"]).sum()) > 5 else -1
        ),
    )
    for s in SITES
])

ref_events = pd.read_csv(OUT / "events.csv")
REQ = ["site", "n", "lo_rate", "hi_rate"]
assert all(c in ref_events.columns for c in REQ), f"out/events.csv lost a required column: {list(ref_events.columns)}"
assert list(events.site) == list(ref_events.site), (
    "site order/names differ from out/events.csv: "
    f"{list(events.site)} vs {list(ref_events.site)}"
)
assert np.array_equal(events.n.to_numpy(), ref_events.n.to_numpy()), "per-site valid-minute counts differ"

for col in ("lo_rate", "hi_rate"):
    d = np.abs(events[col].to_numpy(dtype=float) - ref_events[col].to_numpy(dtype=float))
    assert d.max() <= 1e-9, f"ASSERTION FAILED -- {col}: max |diff| = {d.max():.3e} > 1e-9"
    print(f"  PASS  {col:<8} max|recomputed - out/events.csv| = {d.max():.2e}  (atol 1e-9)")

for col in ("lo_hour_peak", "hi_hour_peak"):
    if col in ref_events.columns:
        same = np.array_equal(events[col].to_numpy(), ref_events[col].to_numpy())
        assert same, f"ASSERTION FAILED -- {col} differs from out/events.csv"
        print(f"  PASS  {col:<13} identical for all {len(events)} sites")
print("  PASS  n            identical for all sites")

cmp = events[REQ].copy()
cmp["lo_rate_ref"] = ref_events.lo_rate.to_numpy()
cmp["hi_rate_ref"] = ref_events.hi_rate.to_numpy()
cmp["|dlo|"] = np.abs(cmp.lo_rate - cmp.lo_rate_ref)
cmp["|dhi|"] = np.abs(cmp.hi_rate - cmp.hi_rate_ref)
print()
print("recomputed vs committed (rates are shares of valid minutes):")
print(cmp.to_string(index=False, float_format=lambda v: f"{v:.12g}"))
print()
print(f"pooled site-average: low {events.lo_rate.mean():.4%} | control {events.hi_rate.mean():.4%} "
      f"| ratio {events.lo_rate.mean() / events.hi_rate.mean():.2f}x "
      f"| sites with low > control {int((events.lo_rate > events.hi_rate).sum())}/{len(events)}")

## 3. Threshold sensitivity sweep (4, 5, 6, 8, 10 dB)

The 6 dB cut is a choice, so the write-up has to show that the *conclusion* does not
depend on it. Both groups are pushed through the identical pipeline at five thresholds;
only the cut moves, the baseline does not.

The committed `out/sensitivity.csv` has columns
`site, n, low_4, low_5, low_6, low_8, low_10, high_4, high_5, high_6, high_8, high_10`.
Note that `low_6` must equal `lo_rate` from the previous section — the two files were
produced by different scripts, so agreement is a genuine cross-check and is asserted.

In [ ]:
sweep = pd.DataFrame([
    dict(
        site=s["site"],
        n=s["n"],
        **{f"low_{t:g}": rate_at(s["lo_excess"], s["lo_valid"], t) for t in SWEEP},
        **{f"high_{t:g}": rate_at(s["hi_excess"], s["hi_valid"], t) for t in SWEEP},
    )
    for s in SITES
])

ref_sweep = pd.read_csv(OUT / "sensitivity.csv")
assert list(sweep.columns) == list(ref_sweep.columns), (
    f"column order differs from out/sensitivity.csv:\n  ours {list(sweep.columns)}\n  ref  {list(ref_sweep.columns)}"
)
assert list(sweep.site) == list(ref_sweep.site), "site order/names differ from out/sensitivity.csv"
assert np.array_equal(sweep.n.to_numpy(), ref_sweep.n.to_numpy()), "per-site counts differ from out/sensitivity.csv"

cols = [c for c in ref_sweep.columns if c not in ("site", "n")]
d = np.abs(sweep[cols].to_numpy(dtype=float) - ref_sweep[cols].to_numpy(dtype=float))
assert d.max() <= 1e-9, f"ASSERTION FAILED -- sweep: max |diff| = {d.max():.3e} > 1e-9"
print(f"  PASS  sensitivity: {d.size} cells ({len(sweep)} sites x {len(cols)} rate columns) "
      f"max|recomputed - out/sensitivity.csv| = {d.max():.2e}  (atol 1e-9)")
print(f"  PASS  sweep.n identical for all {len(sweep)} sites")

cross = np.abs(sweep["low_6"].to_numpy(dtype=float) - events.lo_rate.to_numpy(dtype=float)).max()
assert cross <= 1e-12, f"low_6 must equal lo_rate: max |diff| = {cross:.3e}"
print(f"  PASS  low_6 == lo_rate (two independent scripts): max|diff| = {cross:.2e}")

### The table the paper reports

Rates are unweighted means over the ten sites (a site, not a minute, is the unit of
replication), `ratio` is low / control, and the last column counts how many of the ten
sites show the low band above the control band. The ordering claim in the write-up is
that this count stays at 10/10 across the whole sweep even though the *magnitude* of
both rates falls steeply with the threshold — that is asserted below, because it is a
claim about the data, not a formatting choice.

In [ ]:
tab_rows = []
for t in SWEEP:
    lo_m = float(sweep[f"low_{t:g}"].mean())
    hi_m = float(sweep[f"high_{t:g}"].mean())
    n_gt = int((sweep[f"low_{t:g}"] > sweep[f"high_{t:g}"]).sum())
    tab_rows.append(dict(threshold_dB=int(t), low_rate=lo_m, control_rate=hi_m,
                         ratio_low_over_control=lo_m / hi_m, sites_low_gt_control=n_gt))
tab = pd.DataFrame(tab_rows)

print(f"{'thr (dB)':>9}{'low rate':>10}{'control':>10}{'ratio':>8}{'low>control':>14}")
for r in tab.itertuples():
    print(f"{r.threshold_dB:>9}{r.low_rate:>9.3%}{r.control_rate:>10.3%}"
          f"{r.ratio_low_over_control:>8.2f}{r.sites_low_gt_control:>10d}/{len(sweep)}")

assert (tab.sites_low_gt_control == len(sweep)).all(), (
    "the ordering low > control is NOT invariant across the sweep: "
    f"{tab.sites_low_gt_control.tolist()} of {len(sweep)}"
)
print(f"\n  PASS  ordering invariant: low > control at {len(sweep)}/{len(sweep)} sites for every threshold")
assert (tab.low_rate > tab.control_rate).all(), "mean low rate must exceed mean control rate at every threshold"
assert tab.low_rate.is_monotonic_decreasing, "low-band rate should fall monotonically with the threshold"
print(f"  PASS  magnitude moves: low rate {tab.low_rate.iloc[0]:.2%} at 4 dB -> {tab.low_rate.iloc[-1]:.2%} at 10 dB "
      f"({tab.low_rate.iloc[0] / tab.low_rate.iloc[-1]:.1f}x smaller), ratio "
      f"{tab.ratio_low_over_control.iloc[0]:.2f} -> {tab.ratio_low_over_control.iloc[-1]:.2f}")

## 4. Site x month event rates (the pressure map data)

Same event definition, sliced by calendar month: for every site-month cell, the share of
valid minutes that are event minutes. Months with fewer than 50 valid minutes get no
rate at all (they become blank cells in the figure rather than noisy ones).

Each cell also carries a bootstrap 95% interval over minutes (400 replicates of the
minute set, percentile 2.5/97.5, one fresh `Generator(PCG64)` seeded with 0 per cell).
That makes the interval exactly reproducible — a seeded bootstrap is a deterministic
function of the data — so it is asserted at the same `1e-9` tolerance as the rates,
not just reported.

The committed reference is `out/pressure_events.csv` (135 rows, columns
`site, month, rate, lo, hi, n`, where `n` is the number of valid minutes in that cell).

In [ ]:
def rate_with_ci(event, valid, B=BOOT_B, seed=BOOT_SEED):
    """Share of valid minutes that are event minutes, plus a seeded bootstrap 95% CI."""
    v = event[valid]
    if len(v) < MONTH_MIN_ROWS:
        return np.nan, np.nan, np.nan, len(v)
    r = float(v.mean())
    rng = np.random.default_rng(seed)
    bs = np.array([rng.choice(v, len(v), replace=True).mean() for _ in range(B)], dtype="float64")
    return r, float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5)), len(v)


rows = []
for s in SITES:
    months = pd.Series(s["t"]).dt.to_period("M").astype(str).to_numpy()
    lo_event = (s["lo_excess"] >= THRESH) & s["lo_valid"]
    for month in pd.unique(months):        # first-appearance order == chronological order
        sel = months == month
        r, lo95, hi95, n = rate_with_ci(lo_event[sel], s["lo_valid"][sel])
        if not np.isnan(r):
            rows.append(dict(site=s["site"], month=month, rate=r, lo=lo95, hi=hi95, n=int(n)))
pe = pd.DataFrame(rows)

ref_pe = pd.read_csv(OUT / "pressure_events.csv")
assert list(pe.columns) == list(ref_pe.columns) == ["site", "month", "rate", "lo", "hi", "n"], list(pe.columns)
assert len(pe) == len(ref_pe) == 135, f"row count: recomputed {len(pe)}, committed {len(ref_pe)}, expected 135"
assert list(pe.site) == list(ref_pe.site), "site sequence differs from out/pressure_events.csv"
assert list(pe.month) == list(ref_pe.month), "month sequence differs from out/pressure_events.csv"
assert np.array_equal(pe.n.to_numpy(), ref_pe.n.to_numpy()), "valid-minute counts differ"

for col in ("rate", "lo", "hi"):
    d = np.abs(pe[col].to_numpy(dtype=float) - ref_pe[col].to_numpy(dtype=float))
    assert d.max() <= 1e-9, f"ASSERTION FAILED -- pressure_events.{col}: max |diff| = {d.max():.3e}"
    print(f"  PASS  {col:<4} max|recomputed - out/pressure_events.csv| = {d.max():.2e}  (atol 1e-9, {len(pe)} rows)")
print(f"  PASS  n    identical for all {len(pe)} site-months")
print(f"  PASS  row/key sequence identical ({len(pe)} rows, {pe.site.nunique()} sites)")

pe_out = FIGDIR / "site_month_events.csv"
pe.to_csv(pe_out, index=False)
print(f"\nwrote {pe_out.relative_to(ROOT)} ({len(pe)} rows, {pe_out.stat().st_size / 1024:.1f} kB)")

Per-site summary of the same table. The interval half-width is reported in percentage
points; it is widest where a site has few minutes in a month, which is exactly what the
blank cells in the figure are protecting against.

In [ ]:
print(f"{'site':<20}{'months':>7}{'valid min':>11}{'mean rate':>11}{'CI half-width':>15}{'first':>9}{'last':>9}")
for site, g in pe.groupby("site", sort=False):
    half = float(((g.hi - g.lo) / 2).mean())
    print(f"{site:<20}{len(g):>7}{int(g.n.sum()):>11,}{g.rate.mean():>10.2%}"
          f"{half * 100:>14.2f}pp{g.month.min():>9}{g.month.max():>9}")
print()
print("first five rows:")
print(pe.head(5).to_string(index=False, float_format=lambda v: f"{v:.10g}"))
print("last five rows:")
print(pe.tail(5).to_string(index=False, float_format=lambda v: f"{v:.10g}"))

## 5. Figure 1 — the pressure map

A heatmap of site (rows) by calendar month (columns). The x axis is built from a **full
monthly `period_range`** covering the span of the record, so time is linear in calendar
time: a month in which a site recorded nothing is a visibly blank cell, not a silently
compressed one. Missing cells are `NaN` and are drawn in light grey via the colormap's
"bad" colour; valid cells are coloured by `rate * 100`, i.e. the colourbar is a
percentage.

The colour ceiling is derived from the data (rounded up to an even number) and asserted
to leave no cell clipped. The committed `figs/fig1_pressure_map.png` used a fixed 18 %
ceiling, which saturates the three St Francis cells of the 2015/16 summer (up to 23.1 %);
the regenerated panel keeps them distinguishable instead of flattening them into the
top colour.

No title and no watermark: the caption belongs to the paper, and axis labels plus the
colourbar carry everything the panel needs.

In [ ]:
months = [str(p) for p in pd.period_range(pe.month.min(), pe.month.max(), freq="M")]
sites = sorted(pe.site.unique())
mat = np.full((len(sites), len(months)), np.nan)
mi = {m: i for i, m in enumerate(months)}
si = {s: i for i, s in enumerate(sites)}
for r in pe.itertuples():
    mat[si[r.site], mi[r.month]] = r.rate * 100.0

assert mat.shape == (len(sites), len(months))
assert np.isfinite(mat).sum() == len(pe), "every committed site-month must land in exactly one cell"
assert int(np.isnan(mat).sum()) == mat.size - len(pe), "blank cells must be exactly the unobserved months"
VMAX = float(np.ceil(float(np.nanmax(mat)) / 2.0) * 2.0)
assert np.nanmax(mat) <= VMAX, "colour ceiling must not clip any cell"
print(f"grid: {len(sites)} sites x {len(months)} calendar months ({months[0]} .. {months[-1]})")
print(f"observed cells: {int(np.isfinite(mat).sum())} | blank (no data): {int(np.isnan(mat).sum())}")
print(f"colour range: 0.00% .. {VMAX:.0f}% (data max {np.nanmax(mat):.2f}%, so no cell is clipped)")

SHORT = {
    "01_Richards_Bay": "01 Richards Bay", "02_Durban": "02 Durban",
    "03_Cape_Recife": "03 Cape Recife", "04_Bird_Island": "04 Bird Island",
    "05_St_Croix_Island": "05 St Croix I.", "06_Riy_Banks": "06 Riy Banks",
    "07_St_Francis": "07 St Francis", "09_Fish_Hoek": "09 Fish Hoek",
    "10_Sea_Point": "10 Sea Point", "13_Saldhana_Bay": "13 Saldhana Bay",
}

fig, ax = plt.subplots(figsize=(13.5, 4.6), dpi=300)
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("0.92")
im = ax.imshow(np.ma.masked_invalid(mat), aspect="auto", cmap=cmap,
               vmin=0.0, vmax=VMAX, interpolation="nearest")
ax.set_yticks(range(len(sites)))
ax.set_yticklabels([SHORT.get(s, s) for s in sites], fontsize=9)
step = max(1, len(months) // 14)
ax.set_xticks(range(0, len(months), step))
ax.set_xticklabels([months[i] for i in range(0, len(months), step)],
                   rotation=45, ha="right", fontsize=8)
ax.set_xlabel("calendar month")
ax.set_xticks(np.arange(-0.5, len(months), 1), minor=True)
ax.grid(which="minor", axis="x", color="white", lw=0.35, alpha=0.45)
ax.tick_params(which="minor", length=0)
cb = fig.colorbar(im, ax=ax, pad=0.012)
cb.set_label("% of valid minutes >= 6 dB above running median", fontsize=8.5)
cb.ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _pos: f"{v:.0f}%"))
cb.ax.tick_params(labelsize=8)
fig.tight_layout()
fig1_path = FIGDIR / "fig1_pressure_map.png"
fig.savefig(fig1_path, dpi=300, bbox_inches="tight")
plt.close(fig)
assert fig1_path.is_file() and fig1_path.stat().st_size > 10_000, "figure 1 was not written"
assert plt.imread(fig1_path).shape[0] >= 1000, "figure 1 is unexpectedly small"
print(f"\nwrote {fig1_path.relative_to(ROOT)} "
      f"({fig1_path.stat().st_size / 1024:.0f} kB, 300 dpi, "
      f"{plt.imread(fig1_path).shape[1]}x{plt.imread(fig1_path).shape[0]} px)")

## 6. Figure 2 — threshold sweep: the ordering is invariant, the magnitude is not

Both site-averaged rates are drawn against the threshold on the left axis, and the
low/control ratio on the twin right axis. The point of the panel is the contrast: the
rates fall by roughly an order of magnitude between 4 and 10 dB, while the ratio stays
above one at every threshold and the per-site ordering never inverts (10/10 sites).
That is what lets the write-up treat the 6 dB cut as a presentation choice rather than
a result.

In [ ]:
x = np.asarray(SWEEP, dtype=float)
lo_y = np.array([sweep[f"low_{t:g}"].mean() * 100 for t in SWEEP])
hi_y = np.array([sweep[f"high_{t:g}"].mean() * 100 for t in SWEEP])
ratio_y = lo_y / hi_y

fig, ax = plt.subplots(figsize=(7.8, 4.9), dpi=300)
l1, = ax.plot(x, lo_y, "o-", color="#1f4e79", lw=1.8, ms=6, label="low band (<=794 Hz)")
l2, = ax.plot(x, hi_y, "s-", color="#c55a11", lw=1.8, ms=6, label="control band (>=5 kHz)")
ax.set_xlabel("event threshold (dB above the site's own 181-min running median)")
ax.set_ylabel("event minutes (% of valid minutes)")
ax.set_xticks(x)
ax.set_xticklabels([f"{int(t)}" for t in SWEEP])
ax.set_ylim(0, max(lo_y) * 1.25)
ax.grid(axis="y", color="0.88", lw=0.8)
ax.set_axisbelow(True)

ax2 = ax.twinx()
l3, = ax2.plot(x, ratio_y, "^--", color="0.35", lw=1.4, ms=6,
               label="ratio low / control (right axis)")
ax2.set_ylabel("rate ratio, low band / control band")
ax2.set_ylim(0, max(ratio_y) * 1.45)
ax2.axhline(1.0, color="0.55", lw=0.9, ls=":")

n_gt = tab.sites_low_gt_control.to_numpy()
ax.text(0.03, 0.97,
        f"per-site ordering low > control: {n_gt.min()}/{len(sweep)} at every threshold\n"
        f"rates fall {lo_y[0] / lo_y[-1]:.1f}x from 4 dB to 10 dB; ratio {ratio_y[0]:.2f} -> {ratio_y[-1]:.2f}",
        transform=ax.transAxes, va="top", ha="left", fontsize=7.8, color="0.25")
ax.legend(handles=[l1, l2, l3], frameon=False, fontsize=8.5, loc="center right")
fig.tight_layout()
fig2_path = FIGDIR / "fig2_threshold_sweep.png"
fig.savefig(fig2_path, dpi=300, bbox_inches="tight")
plt.close(fig)
assert fig2_path.is_file() and fig2_path.stat().st_size > 10_000, "figure 2 was not written"
assert (ratio_y > 1).all(), "ratio must stay above 1 at every threshold"
print(f"wrote {fig2_path.relative_to(ROOT)} "
      f"({fig2_path.stat().st_size / 1024:.0f} kB, 300 dpi, "
      f"{plt.imread(fig2_path).shape[1]}x{plt.imread(fig2_path).shape[0]} px)")
print()
print(f"{'thr':>5}{'low %':>8}{'control %':>11}{'ratio':>7}{'low>control':>13}")
for t, a, b, r in zip(SWEEP, lo_y, hi_y, ratio_y):
    print(f"{int(t):>5}{a:>8.2f}{b:>11.2f}{r:>7.2f}{int((sweep[f'low_{t:g}'] > sweep[f'high_{t:g}']).sum()):>9d}/{len(sweep)}")

## 7. Reproducing the submission: solving the test set's moments

The competition metric is `score = 1 - RMSE`. For a **constant** prediction `c` the RMSE
is exactly `sqrt(s^2 + (m - c)^2)`, where `m` and `s` are the mean and standard deviation
of the truth on the public split. So each submitted constant and its public score are
one equation in the two unknowns.

Lifting the square gives something linear: with `r_i = (1 - score_i)^2 - c_i^2` and
`k = s^2 + m^2`,

$$ r_i = k - 2 m\,c_i . $$

That is a two-column least-squares problem in `(k, m)`; solving it over **all** pairs in
`const_points.json` gives `m` directly and `s = sqrt(k - m^2)`. With the two pairs
currently on file the system is exactly determined, so the least-squares solution is the
closed-form pairwise solution — the notebook verifies that equivalence explicitly.

The optimal constant is then `c* = m` (it is the value that zeroes the bias term), and
its predicted score is `1 - s`.

In [ ]:
points = json.loads((ROOT / "const_points.json").read_text())
assert isinstance(points, list) and len(points) >= 2, f"need at least two (constant, score) pairs, got {points}"
assert all(len(p) == 2 for p in points), "each entry must be [constant, public_score]"

c_pts = np.array([float(p[0]) for p in points], dtype="float64")
score_pts = np.array([float(p[1]) for p in points], dtype="float64")
rhs = (1.0 - score_pts) ** 2 - c_pts ** 2

A = np.column_stack([np.ones_like(c_pts), -2.0 * c_pts])
assert np.linalg.matrix_rank(A) == 2, "the constants must not all be identical (rank-deficient system)"
sol, *_ = np.linalg.lstsq(A, rhs, rcond=None)
k_hat, m_hat = float(sol[0]), float(sol[1])
s_hat = math.sqrt(max(0.0, k_hat - m_hat ** 2))

# Independent check 1: the closed form for a single pair must agree.
c1, c2 = c_pts[0], c_pts[-1]
r1, r2 = 1.0 - score_pts[0], 1.0 - score_pts[-1]
m_pair = (r1 * r1 - r2 * r2 - c1 * c1 + c2 * c2) / (2.0 * (c2 - c1))
assert abs(m_pair - m_hat) <= 1e-9, f"least-squares m={m_hat!r} vs pairwise closed form m={m_pair!r}"
print(f"  PASS  least-squares m == pairwise closed-form m (|diff| = {abs(m_pair - m_hat):.2e})")

# Independent check 2: the solution must reproduce every observed score.
pred = 1.0 - np.sqrt(s_hat ** 2 + (m_hat - c_pts) ** 2)
resid = np.abs(pred - score_pts).max()
assert resid <= 1e-9, f"ASSERTION FAILED -- solved moments do not reproduce the observed scores: {resid:.3e}"
print(f"  PASS  all {len(points)} observed scores reproduced, max|residual| = {resid:.2e}")

print()
print(f"points        : {[(float(c), float(s_)) for c, s_ in zip(c_pts, score_pts)]}")
print(f"m (truth mean): {m_hat!r}")
print(f"s (truth sd)  : {s_hat!r}")
print(f"k = s^2 + m^2 : {k_hat!r}")
print(f"optimal const : c* = m = {m_hat!r}")
print(f"predicted score at c* : 1 - s = {1.0 - s_hat!r}")
print(f"RMSE at c*    : {s_hat!r}")
print()
print(f"{'constant':>10}{'observed':>14}{'predicted':>14}{'residual':>12}")
for c, sc, pr in zip(c_pts, score_pts, pred):
    print(f"{c:>10.4f}{sc:>14.9f}{pr:>14.9f}{pr - sc:>12.2e}")

### The optimal-constant submission file

A constant submission is the honest baseline the write-up reports: it fixes the level of
the prediction but carries no per-row information. The file is written into `PACKAGE/`
(not the top level) so it cannot be confused with an actually-submitted file, and its ID
order is taken from `dataset/sample_submission.csv`.

In [ ]:
test = pd.read_csv(ROOT / "dataset" / "Test.csv")
sample = pd.read_csv(ROOT / "dataset" / "sample_submission.csv")
assert len(test) == 1500 and test.shape[1] == 376, f"unexpected Test.csv shape {test.shape}"
assert list(sample.columns) == ["ID", "share_added_vessel", "score_added_vessel"], list(sample.columns)
assert len(sample) == 1500 and sample.ID.is_unique, "sample_submission must have 1500 unique IDs"
assert set(test.ID) == set(sample.ID), "Test.csv and sample_submission.csv disagree on the ID set"

order_matches = list(test.ID) == list(sample.ID)
print(f"Test.csv: {test.shape} | sample_submission.csv: {sample.shape}")
print(f"ID set equal  : {set(test.ID) == set(sample.ID)}")
print(f"ID order equal: {order_matches}  (Test.csv order {'matches' if order_matches else 'DIFFERS from'} sample_submission.csv)")
assert order_matches, "Test.csv ID order differs from sample_submission.csv; the file below follows sample order"
assert np.isfinite(m_hat), "optimal constant must be finite"

submission = sample.copy()
submission["share_added_vessel"] = m_hat
submission["score_added_vessel"] = m_hat
sub_path = PKG / "submission_optimal_constant.csv"
submission.to_csv(sub_path, index=False)

assert len(submission) == 1500, f"submission has {len(submission)} rows, expected 1500"
assert submission.ID.is_unique and submission.notna().all().all(), "submission has duplicate IDs or NaNs"
assert np.allclose(submission.share_added_vessel.to_numpy(dtype=float), m_hat, atol=0, rtol=0)
assert np.allclose(submission.score_added_vessel.to_numpy(dtype=float), m_hat, atol=0, rtol=0)

reread = pd.read_csv(sub_path)
assert len(reread) == 1500 and list(reread.columns) == ["ID", "share_added_vessel", "score_added_vessel"]
assert list(reread.ID) == list(sample.ID), "written file does not preserve sample_submission ID order"
print(f"\nwrote {sub_path.relative_to(ROOT)} ({sub_path.stat().st_size / 1024:.1f} kB, 1500 rows)")
print(f"  PASS  1500 rows | IDs unique | both prediction columns constant at c* = {m_hat:.9f}")
print(f"  PASS  ID order of the written file == sample_submission.csv order")
print()
print("first rows:")
print(submission.head(5).to_string(index=False, float_format=lambda v: f"{v:.9f}"))

## 8. The submitted model vector, and a counterfactual that was *not* submitted

The best public score the project actually obtained came from an inverted,
three-way-ensemble prediction vector (`inv/submission_inv_mix3_ens.csv`), whose public
leaderboard score was **0.795607712**. That vector is read here, never rewritten.

Because the metric decomposes into a bias term and a shape term, the same moment
solution from section 7 lets us ask what that vector would have scored **had it been
re-centred to the truth's mean `m`**:

$$ \text{rmse}_{\text{new}} = \sqrt{\max\left(0, (1-\text{score})^2 - (\bar y_{\text{old}} - m)^2\right)} . $$

This treats the error as a Pythagorean decomposition into a mean offset and a residual
shape error; it is a *counterfactual*, computed from published quantities only, and it
is **not** a submission. No file is written in this section, by design.

In [ ]:
LB_SUBMITTED = 0.795607712          # public leaderboard score of the vector below
submitted = pd.read_csv(ROOT / "inv" / "submission_inv_mix3_ens.csv")
assert len(submitted) == 1500, f"submitted vector has {len(submitted)} rows"
assert "share_added_vessel" in submitted.columns, list(submitted.columns)
assert list(submitted.ID) == list(sample.ID), "submitted vector ID order differs from sample_submission"

vec = submitted.share_added_vessel.to_numpy(dtype="float64")
assert np.isfinite(vec).all(), "submitted vector contains non-finite values"
mean_old = float(vec.mean())
sd_old_ddof1 = float(vec.std(ddof=1))
sd_old_ddof0 = float(vec.std(ddof=0))

rmse_obs = 1.0 - LB_SUBMITTED
bias2 = (mean_old - m_hat) ** 2
radicand = rmse_obs ** 2 - bias2
assert radicand > 0, (
    "the Pythagorean decomposition does not hold: (1-score)^2 <= (mean_old - m)^2 "
    f"({rmse_obs ** 2:.6f} <= {bias2:.6f})"
)
new_rmse = math.sqrt(radicand)
cf_score = 1.0 - new_rmse

print("submitted vector: inv/submission_inv_mix3_ens.csv  (read-only)")
print(f"  n            : {len(submitted)}")
print(f"  mean         : {mean_old!r}")
print(f"  sd (ddof=1)  : {sd_old_ddof1!r}")
print(f"  sd (ddof=0)  : {sd_old_ddof0!r}")
print(f"  min / max    : {vec.min():.6f} / {vec.max():.6f}")
print(f"  public score : {LB_SUBMITTED}   => observed RMSE {rmse_obs:.6f}")
print()
print(f"recovered truth mean m = {m_hat!r}")
print(f"mean offset (mean_old - m) = {mean_old - m_hat:.6f}  => bias^2 = {bias2:.6f}")
print(f"observed RMSE^2            = {rmse_obs ** 2:.6f}")
print(f"shape error after re-centring = sqrt(rmse^2 - bias^2) = {new_rmse:.6f}")
print()
print(f"COUNTERFACTUAL (NOT SUBMITTED): re-centred to m, score would be {cf_score:.6f}"
      f"  (+{cf_score - LB_SUBMITTED:.6f} vs the submitted vector)")
print(f"  implied R^2 against the truth's spread: 1 - (shape/s)^2 = {1.0 - (new_rmse / s_hat) ** 2:.3f}")
print(f"  a constant at the vector's own mean would score "
      f"{1.0 - math.sqrt(s_hat ** 2 + bias2):.6f} — the vector beats it, so it carries real signal")

assert cf_score > LB_SUBMITTED, (
    f"re-centring should improve the score: counterfactual {cf_score:.6f} vs actual {LB_SUBMITTED}"
)
assert cf_score > 1.0 - math.sqrt(s_hat ** 2 + bias2), (
    "the vector must beat a constant placed at its own mean, otherwise it carries no shape signal"
)
print("\n  PASS  counterfactual > submitted score, and > constant-at-own-mean")
print("  NOT SUBMITTED: this is an arithmetic counterfactual, no file was written")

## 9. What this notebook reproduced, and what it read

**Recomputed from the raw parquets** (no committed number is used as an input):

* the 10 site-level 6 dB exceedance rates and their valid-minute counts
  (`out/events.csv`, asserted at `1e-9`);
* the full 4/5/6/8/10 dB sweep for both band groups (`out/sensitivity.csv`, asserted at
  `1e-9`), and the site-averaged table with the 10/10 ordering claim;
* the 135 site-month rates **and their seeded bootstrap intervals**
  (`out/pressure_events.csv`, asserted at `1e-9`) — re-written to
  `PACKAGE/figs_regen/site_month_events.csv`;
* both figures, from the recomputed table, at 300 dpi:
  `PACKAGE/figs_regen/fig1_pressure_map.png`,
  `PACKAGE/figs_regen/fig2_threshold_sweep.png`;
* the test-set moments `(m, s)`, the optimal constant `c* = m`, its predicted score
  `1 - s`, and `PACKAGE/submission_optimal_constant.csv` (1500 rows);
* the re-centring counterfactual for the submitted vector.

**Read as reference targets or as inputs that cannot be recomputed offline:**

* `out/events.csv`, `out/sensitivity.csv`, `out/pressure_events.csv` — the committed
  artifacts used *only* to assert equality;
* `const_points.json` — two (constant, public score) pairs measured on the live
  leaderboard; these are observations, not something a local run can regenerate;
* `inv/submission_inv_mix3_ens.csv` — the already-submitted prediction vector, and
  `0.795607712`, its public score, which is likewise a leaderboard observation.

Everything else — the figures, the moment solution, the submission file — is derived
here from `dataset/` and those observations alone.

**Runtime.** The pipeline is dominated by reading ~185 MB of parquet and by the 135
seeded bootstraps (400 resamples of up to ~27k minutes each). The final cell prints the
exact wall-clock time of *this* run; on the reference machine it is tens of seconds.

In [ ]:
TOTAL_SECONDS = time.perf_counter() - T_START
FP_AFTER = fingerprint(READONLY)

changed = sorted(k for k in FP_BEFORE if FP_BEFORE[k] != FP_AFTER.get(k))
added = sorted(set(FP_AFTER) - set(FP_BEFORE))
assert not changed, f"READ-ONLY VIOLATION: these files changed: {changed}"
assert not added, f"READ-ONLY VIOLATION: new files appeared in guarded locations: {added}"
print(f"  PASS  provenance guard: {len(FP_AFTER)} read-only files byte-identical "
      f"(sha256) before and after this run")

print()
print("=" * 78)
print("REPRODUCTION SUMMARY")
print("=" * 78)
print(f"raw input              : {len(SITES)} site parquets, {sum(s['n'] for s in SITES):,} valid minutes")
print(f"events.csv             : {len(events)} sites, max|diff| <= 1e-9")
print(f"sensitivity.csv        : {len(sweep)} sites x {len(SWEEP)} thresholds x 2 groups, max|diff| <= 1e-9")
print(f"pressure_events.csv    : {len(pe)} site-months (rate, lo, hi, n), max|diff| <= 1e-9")
print(f"figures                : {fig1_path.name}, {fig2_path.name} (300 dpi) -> {FIGDIR.name}/")
print(f"test-set moments       : m = {m_hat!r}")
print(f"                         s = {s_hat!r}")
print(f"optimal constant       : c* = {m_hat!r} (predicted score {1.0 - s_hat!r})")
print(f"submission file        : {(PKG / 'submission_optimal_constant.csv').relative_to(ROOT)} (1500 rows)")
print(f"submitted vector score : {LB_SUBMITTED}")
print(f"counterfactual (NOT submitted): {cf_score!r}")
print(f"wall clock             : {TOTAL_SECONDS:.1f} s")
print("=" * 78)
print("done")